# Batching and Performance

## What you'll learn

- Understand the two-level batching model (artifacts → execution units → workers)
- Configure `ExecutionConfig` fields for different workload patterns
- Measure the impact of batching on pipeline performance
- Choose batching parameters based on workload characteristics

**Prerequisites:** [Advanced Patterns](../pipeline-patterns/05-advanced-patterns.ipynb)
(for batching intro), [First Pipeline](../getting-started/01-first-pipeline.ipynb).  
**Estimated time:** 20 minutes  
**GPU required:** No.

---

The [Advanced Patterns](../pipeline-patterns/05-advanced-patterns.ipynb) tutorial
introduced `artifacts_per_unit` as a way to group work. This tutorial goes
deeper: you'll learn the full two-level batching model, experiment with all
`ExecutionConfig` fields, and develop intuition for tuning them.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import Backend, PipelineManager
from artisan.utils import tutorial_setup
from artisan.utils.logging import configure_logging
from artisan.visualization import inspect_pipeline
from artisan.visualization.timing import PipelineTimings

configure_logging()

In [ ]:
env = tutorial_setup("batching_performance")

## The two-level batching model

Artisan batches work in two stages:

```
Input artifacts         Execution units              Workers
┌───┐ ┌───┐ ┌───┐     ┌─────────────┐             ┌──────────┐
│ a │ │ b │ │ c │ ──→  │ unit 0      │ ──→         │ worker 0 │
└───┘ └───┘ └───┘      │ [a, b, c]   │             │ unit 0   │
                        └─────────────┘             │ unit 1   │
┌───┐ ┌───┐ ┌───┐     ┌─────────────┐             └──────────┘
│ d │ │ e │ │ f │ ──→  │ unit 1      │
└───┘ └───┘ └───┘      │ [d, e, f]   │             ┌──────────┐
                        └─────────────┘     ──→     │ worker 1 │
┌───┐ ┌───┐ ┌───┐     ┌─────────────┐             │ unit 2   │
│ g │ │ h │ │ i │ ──→  │ unit 2      │             │ unit 3   │
└───┘ └───┘ └───┘      │ [g, h, i]   │             └──────────┘
                        └─────────────┘
┌───┐ ┌───┐ ┌───┐     ┌─────────────┐
│ j │ │ k │ │ l │ ──→  │ unit 3      │
└───┘ └───┘ └───┘      │ [j, k, l]   │
                        └─────────────┘
```

**Level 1: Artifacts → Execution units.** Controlled by `artifacts_per_unit`.
Each execution unit is one call to the operation's `execute()` method.

**Level 2: Execution units → Workers.** Controlled by `units_per_worker` and
`max_workers`. Each worker is one process (local) or one SLURM job (submitted).
Multiple units can run sequentially within one worker.

## ExecutionConfig fields

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `artifacts_per_unit` | `int` | `1` | Artifacts grouped into each execution unit |
| `max_artifacts_per_unit` | `int \| None` | `None` | Cap on artifacts per unit (for variable-size inputs) |
| `units_per_worker` | `int` | `1` | Execution units assigned to each worker process/job |
| `max_workers` | `int \| None` | `None` | Maximum concurrent workers (`None` = no limit) |
| `estimated_seconds` | `float \| None` | `None` | Hint for SLURM time allocation |
| `job_name` | `str \| None` | `None` | Custom SLURM job name (defaults to operation name) |

Pass these as a dict to the `execution` parameter of `pipeline.run()`.

## Baseline: default batching

First, run a pipeline with the default configuration (1 artifact per unit)
to establish a timing baseline.

In [ ]:
pipeline = PipelineManager.create(
    name="baseline",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    backend=Backend.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env.delta_root)

In [ ]:
baseline_timings = PipelineTimings.from_delta(env.delta_root)
baseline_timings.step_timings()

With default batching, step 1 (DataTransformer) creates 12 execution
units — one per artifact. Each unit has its own startup overhead (process
creation, artifact loading). The step timings show the total cost.

## Tuning `artifacts_per_unit`

Group multiple artifacts into fewer execution units to reduce per-unit
overhead. This is the most common tuning parameter.

In [ ]:
env_apu = tutorial_setup("batching_apu")

pipeline = PipelineManager.create(
    name="apu_tuning",
    delta_root=env_apu.delta_root,
    staging_root=env_apu.staging_root,
    working_root=env_apu.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    execution={"artifacts_per_unit": 4},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    execution={"artifacts_per_unit": 6},
    backend=Backend.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_apu.delta_root)

In [ ]:
apu_timings = PipelineTimings.from_delta(env_apu.delta_root)
apu_timings.step_timings()

Step 1 now has 3 execution units (12 artifacts / 4 per unit) instead of 12.
Step 2 has 2 units (12 / 6). Fewer units means less overhead from process
creation and artifact serialization. The execution phase may be slightly
longer per unit (more work per call), but the total time is usually lower.

## Tuning `units_per_worker`

On SLURM clusters, each worker is a separate job submission. Job startup has
significant overhead (queue wait, container launch, environment setup).
`units_per_worker` packs multiple execution units into a single job to
amortize this cost.

Locally, `units_per_worker` packs units into a single process. This reduces
process creation overhead but runs the units sequentially within that process.

In [ ]:
env_upw = tutorial_setup("batching_upw")

pipeline = PipelineManager.create(
    name="upw_tuning",
    delta_root=env_upw.delta_root,
    staging_root=env_upw.staging_root,
    working_root=env_upw.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    execution={"artifacts_per_unit": 4, "units_per_worker": 3},
    backend=Backend.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_upw.delta_root)

With `artifacts_per_unit=4` and 12 inputs, we get 3 execution units. With
`units_per_worker=3`, all 3 units are packed into 1 worker. On SLURM, this
means 1 job instead of 3. Locally, it means 1 process running all 3 units
sequentially.

The primary benefit is on SLURM: fewer job submissions means less queue wait
time and container startup overhead. Locally the benefit is smaller — it
mainly reduces process creation cost.

## Limiting parallelism with `max_workers`

When running on shared resources (GPU cluster, limited memory), you may need
to cap the number of concurrent workers. Set `max_workers` to limit parallelism.

In [ ]:
env_mw = tutorial_setup("batching_max_workers")

pipeline = PipelineManager.create(
    name="max_workers_tuning",
    delta_root=env_mw.delta_root,
    staging_root=env_mw.staging_root,
    working_root=env_mw.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 12, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    execution={"artifacts_per_unit": 2, "max_workers": 2},
    backend=Backend.LOCAL,
)

pipeline.finalize()
inspect_pipeline(env_mw.delta_root)

With `artifacts_per_unit=2` and 12 inputs, we get 6 execution units. With
`max_workers=2`, only 2 units run concurrently — the rest wait in a queue.
Use this when your operation is resource-intensive (e.g., GPU inference) and
you cannot afford to run many instances simultaneously.

## Operation-level defaults

Operations can declare their own `ExecutionConfig` defaults in the class
definition. Step-level overrides take precedence.

```python
class MyExpensiveOperation(OperationDefinition):
    name = "my_expensive_op"

    # Declare default batching for this operation
    execution = ExecutionConfig(
        artifacts_per_unit=10,
        max_workers=4,
    )
```

When you call `pipeline.run(MyExpensiveOperation)` without an `execution`
override, these defaults are used. If you pass `execution={"max_workers": 2}`,
the step override wins.

## Tuning guidelines

| Workload type | `artifacts_per_unit` | `units_per_worker` | `max_workers` |
|---------------|---------------------|-------------------|---------------|
| **Fast, many artifacts** (e.g., metrics) | 10-50 | 1 | Default |
| **Slow, few artifacts** (e.g., structure prediction) | 1 | 1 | Limited by GPU count |
| **SLURM with container startup** | 5-20 | 3-10 | Default |
| **Memory-intensive** | 1-5 | 1 | Limited by available memory |
| **I/O-bound** | 10-100 | 1 | Default (I/O parallelism helps) |

**Rules of thumb:**

1. Start with defaults and measure with `PipelineTimings`
2. If `execute` phase dominates and per-unit time is short, increase `artifacts_per_unit`
3. If SLURM queue wait dominates, increase `units_per_worker`
4. If resource contention is an issue, decrease `max_workers`
5. Use `max_artifacts_per_unit` when input sizes vary and you want to cap memory usage

## Summary

| Concept | Parameter | Effect |
|---------|-----------|--------|
| Artifact grouping | `artifacts_per_unit` | Fewer execution units, less per-unit overhead |
| Unit cap | `max_artifacts_per_unit` | Prevents oversized units with variable inputs |
| Job packing | `units_per_worker` | Fewer SLURM jobs, amortized startup cost |
| Parallelism limit | `max_workers` | Caps concurrent processes/jobs for resource control |
| Time hint | `estimated_seconds` | SLURM time allocation (no effect locally) |
| Job naming | `job_name` | Custom SLURM job name for monitoring |

**Key takeaway:** Start with defaults, measure with `PipelineTimings`, and
tune the parameter that targets your specific bottleneck. `artifacts_per_unit`
is almost always the first knob to turn.

## Next steps

- [Step Overrides](02-step-overrides.ipynb) — All `pipeline.run()` override parameters
- [Timing Analysis](../working-with-results/04-timing-analysis.ipynb) — Diagnose performance with timing DataFrames
- [Configuring Execution](../../how-to-guides/configuring-execution.md) — Complete execution configuration reference